# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [1]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app)
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

# Limitación observable de S5.
limitacion_s5 = (
    'El resultado solo existe en la ejecución que pulsa el botón: al hacer un '
    'rerun (otro widget, R) desaparece, no hay estado de carga ni de error '
    'explícito y no se puede reintentar ni limpiar.'
)
print('Limitación observada:', limitacion_s5)

app de S5: C:\Users\pelay\Documents\COMILLAS\MLOPS\workspace\icai-muiaap-operacion-de-modelos\semana5\modules\05-streamlit-basic-model-ui\solutions\02-first-streamlit\app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: El resultado solo existe en la ejecución que pulsa el botón: al hacer un rerun (otro widget, R) desaparece, no hay estado de carga ni de error explícito y no se puede reintentar ni limpiar.


## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Tipo | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- | --- |
| `last_state` | `UiState` | `st.session_state` | Sí | Sí, vuelve a `UiState(phase="idle")` | Permite volver a pintar el último resultado o error tras un rerun. |
| `last_values` | `dict[str, float]` | `st.session_state` | Sí | Sí, vuelve a `{}` | Solo sirve para reintentar el último envío; no se muestra ni se registra. |
| `telemetry` | `Telemetry` | `st.session_state` | Sí | No | Acumula contadores, códigos de error, latencias y versiones de la sesión; limpiar la pantalla no borra la historia operativa. Nunca guarda features. |
| gateway / bundle | `InferenceGateway` | `st.cache_resource` (argumento: ruta del bundle) | Sí, y se comparte entre sesiones | No | Cargar el bundle es caro, así que se carga una vez. Es global: debe ser de solo lectura y seguro en concurrencia. |
| respuesta de una petición | `PredictionPayload` → `PredictionView` | Variable local de esa ejecución, y después dentro de `last_state` | No por sí misma (sí a través de `last_state`) | Sí, junto con `last_state` | Pertenece a una persona y a una petición: nunca se cachea como recurso compartido. |

Referencias: [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) y [`st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource).

## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [2]:
# Componente y justificación de cada zona de la interfaz.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {
        'component': 'st.form',
        'reason': 'Heredado de S5: agrupa los 11 campos y solo infiere al pulsar el botón.',
    },
    'estado_operacion': {
        'component': 'st.status',
        'reason': 'Comunica la operación en curso (loading) y cómo terminó, sin mover el formulario.',
    },
    'resumen_resultado': {
        'component': 'st.metric',
        'reason': 'Tres métricas en columnas: categoría, confianza y latencia se leen de un vistazo.',
    },
    'mensaje_interpretacion': {
        'component': 'st.warning',
        'reason': 'Avisa de confianza baja o latencia alta sin convertirlo en error; st.info si no hay aviso.',
    },
    'trazabilidad': {
        'component': 'st.expander',
        'reason': 'Versiones y request_id quedan disponibles pero plegados, sin invadir el resultado.',
    },
    'separacion_telemetria': {
        'component': 'st.divider',
        'reason': 'Separa el resultado de la persona de la telemetría agregada de la sesión.',
    },
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
assert pending == [] and invalid == []
print('Comprobación final: OK')

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: OK


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

**Cambio respecto al plano:** ninguno en la estructura. El cambio visual hecho en el laboratorio (`ui_playground.py`) es quitar el `st.success(quality_label)` que aparecía encima de las métricas en los escenarios de éxito. Repetía la categoría y la pintaba en verde incluso cuando era «Requiere revisión», lo que sugiere un resultado bueno. Ahora la categoría solo aparece en `st.metric` y el color queda para los avisos de confianza y latencia.


## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.

| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- | --- |
| `idle` | Inicio de sesión / «Limpiar último resultado» | Resultado | `st.info` con la instrucción de rellenar el formulario y pulsar «Ejecutar inferencia» | Editar el formulario y enviarlo | `last_state.phase == "idle"` y `last_values == {}`; la telemetría conserva sus contadores |
| `loading` | Pulsar «Ejecutar inferencia» | Operación | `st.status` «Ejecutando inferencia…» | Esperar; el formulario sigue visible | El controlador emite `loading` con su `request_id` antes de llamar a `gateway.predict` |
| `success` | El gateway devuelve un `PredictionPayload` válido | Resultado y trazabilidad | Tres `st.metric`; aviso de confianza; aviso de lentitud si supera 300 ms; versiones y `request_id` en `st.expander` | Enviar otra muestra o limpiar | `last_state.view` con `confidence_level` y `latency_status`; `requests_success` +1 |
| `error` | Excepción controlada del gateway | Resultado | `st.error` con título y mensaje seguros, «Qué puedes hacer» y `request_id` | Reintentar con `last_values`, corregir y reenviar, o limpiar | `last_state.error.code`; `errors_by_code[code]` +1 |

**Recuperación por tipo de error:**

| Error | Código | Recuperación que se ofrece |
| --- | --- | --- |
| Entrada inválida | `invalid_input` | Revisar los valores del formulario y volver a enviarlo. |
| Bundle no disponible | `artifact_unavailable` | Configurar `MODEL_UI_BUNDLE` con un bundle válido y reiniciar; reintentar no lo arregla. |
| Timeout | `timeout` | Reintentar el mismo envío con «Reintentar». |

**Lo que la persona nunca ve:** traceback, rutas internas del bundle, secretos, el mensaje original de la excepción ni el payload completo de features.

## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [3]:
ux_copy = {
    'low_confidence': 'Confianza baja: el resultado es orientativo y requiere revisión; la confianza no es una garantía.',
    'slow_success': 'La respuesta llegó, pero la latencia supera el objetivo técnico de la demo.',
    'invalid_input': 'Algún valor del formulario no cumple el contrato del modelo; revísalo y vuelve a enviarlo.',
    'artifact_unavailable': 'No se pudo cargar el modelo en este momento; configura un bundle válido y reinicia la app.',
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
assert pending_copy == []
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?
- ¿Qué función cargará el gateway una sola vez y cuál será su límite?
- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?
- ¿Qué ve una persona con confianza baja o una respuesta lenta?
- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

### Respuestas y plan de implementación

- **Otro widget frente a «Limpiar»:** un rerun conserva `last_state`, `last_values` y `telemetry` en `st.session_state`. «Limpiar» devuelve `last_state` a `idle` y vacía `last_values`, pero deja la telemetría intacta.
- **Gateway cargado una vez:** `get_cached_gateway()` envuelve `build_gateway(bundle_value)` con `st.cache_resource`. Su límite: es un recurso global compartido entre sesiones, así que solo depende de la ruta del bundle y no guarda datos de ninguna petición.
- **`render_state()`:** `loading` → `st.status`; `success` → tres `st.metric` en `st.columns`, `st.warning`/`st.info` para confianza y latencia y `st.expander` de trazabilidad; `error` → `st.error`, recuperación y `request_id` en `st.caption`.
- **Confianza baja o respuesta lenta:** el resultado sigue en `success`; aparece un `st.warning` de confianza baja y, aparte, otro de lentitud si supera 300 ms.
- **`request_id` y versiones:** en el `st.expander` «Versiones y petición» (éxito) o en un `st.caption` bajo el error.

| Requisito | Función del taller (6.2) | Test que lo comprueba |
| --- | --- | --- |
| Inicialización idempotente de la sesión | `session.initialize_session_state` | `test_session_state_initialization_is_idempotent` |
| Limpiar sin perder telemetría | `session.clear_last_state` | `test_clear_last_state_keeps_telemetry` |
| Umbrales de confianza (0.60 / 0.85) | `policies.classify_confidence` | `test_confidence_policy_has_explicit_boundary_values` |
| Objetivo de latencia (300 ms) | `policies.classify_latency` | `test_latency_and_confidence_are_explained_to_the_user` |
| Vista con copy de confianza y latencia | `presentation.build_prediction_view` | `test_latency_and_confidence_are_explained_to_the_user` |
| Errores sin detalle técnico | `presentation.to_user_facing_error` | `test_error_translation_does_not_expose_technical_detail` |
| `loading` → `success` con versiones en telemetría | `controller.PredictionController.submit` | `test_controller_emits_loading_then_success_and_records_versions` |
| `loading` → `error` sin payload en telemetría | `controller.PredictionController.submit` | `test_controller_emits_loading_then_error_without_payload_telemetry` |
| Gateway cacheado, estado persistente, reintentar y limpiar | `app.get_cached_gateway`, `app.render_state`, `app.main` | Revisión visual en la app |


## Debrief

- ¿Qué limita a S5 aunque el formulario ya funcione?
- ¿Por qué el gateway es un recurso cacheable y el resultado de una petición pertenece a una sesión?
- ¿Qué componente hace visible cada transición sin añadir una segunda lógica de inferencia?
- ¿Por qué categoría, confianza y latencia no tienen el mismo significado?
- ¿Qué parte de este diseño se conserva al sustituir el gateway local por HTTP en S7?